# Réplica del experimento del paper y validación cronológica

Este notebook hace dos cosas con `data/data_cajica_2022_crudo.csv`:

1. **Réplica**: reconstruye el pipeline de `V_G1.ipynb` (las 8 variables reales, discretización de viento y temperatura,
   re-muestreo a 1 min con relleno hacia atrás, muestreo aleatorio y 10-fold) para comprobar que se reproduce el orden de magnitud del paper (R² ≈ 0.80).
2. **Validación cronológica**: repite la comparación de los 20 modelos con folds *en el tiempo* (ventana creciente, con un día de separación)
   y evaluando solo sobre observaciones reales (sin las filas replicadas por el relleno). De aquí salen las **Tablas II y V** para reemplazar en el paper.

Diferencias respecto al original (PyCaret 2.3.10) que conviene declarar: se usa scikit-learn, XGBoost, LightGBM y CatBoost con parámetros por defecto
y `random_state=123`; `MAPE` se calcula solo sobre `V_G1 > 0` (con ceros la fórmula estándar no está definida).

## 0. Preparación

In [1]:
from pathlib import Path
import time, json, math, warnings, platform
import numpy as np, pandas as pd
import sklearn
warnings.filterwarnings("ignore")
from sklearn.model_selection import KFold, TimeSeriesSplit, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import (LinearRegression, Ridge, BayesianRidge, Lars, OrthogonalMatchingPursuit,
                                  HuberRegressor, ElasticNet, Lasso, LassoLars, PassiveAggressiveRegressor)
from sklearn.dummy import DummyRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

BASE = Path.cwd(); DATA = BASE / "data"; OUT = BASE / "outputs"; OUT.mkdir(exist_ok=True)
SEED = 123
print(platform.python_version(), "sklearn", sklearn.__version__, "pandas", pd.__version__)

3.11.15 sklearn 1.9.1 pandas 3.1.0


## 1. Reconstruir el pipeline del paper

Reglas tomadas de `V_G1.ipynb`: valores fuera de límites físicos → 0 (viento > 65 m/s, `xejz` > 8, `V_G1` > 25 V);
humedad relativa fuera de 30–70 % → media; serie a rejilla de 1 min con `bfill`; viento y temperatura discretizados
(`xmag_disc` = 0 si < 3.5 m/s, `TEMP_norm` = 0 si < 23 °C).

In [2]:
COLS = ['ID','velU','velV','velW','xang','xejz','xmag','V_G1','FR_G1','P_AT','EG1','V_G2','FR_G2','P_AT2','EG2',
        'HUMEDAD_SUELO','HUMEDAD_RELATIVA','TEMPERATURA','fecha']
raw = pd.read_csv(DATA / "data_cajica_2022_crudo.csv", names=COLS, header=None, parse_dates=['fecha'])
df = raw.copy()
df.loc[df.xmag > 65, 'xmag'] = 0
df.loc[df.xejz > 8, 'xejz'] = 0
df.loc[df.V_G1 > 25, 'V_G1'] = 0
df.loc[df.HUMEDAD_RELATIVA > 70, 'HUMEDAD_RELATIVA'] = df.HUMEDAD_RELATIVA.mean()
df.loc[df.HUMEDAD_RELATIVA < 30, 'HUMEDAD_RELATIVA'] = df.HUMEDAD_RELATIVA.mean()

real = df.drop_duplicates('fecha', keep='last').sort_values('fecha').set_index('fecha')
grid = real.asfreq('60s', method='bfill')
grid['is_real'] = grid.index.isin(real.index)
grid['hour'] = grid.index.hour
grid['date_h'] = (grid.index.floor('h') - grid.index.min()).total_seconds() / 3600.0
grid['xmag_disc'] = grid['xmag'].where(grid['xmag'] >= 3.5, 0.0)
grid['TEMP_norm'] = grid['TEMPERATURA'].where(grid['TEMPERATURA'] >= 23, 0.0)

FEATURES = ['xang','xejz','HUMEDAD_SUELO','HUMEDAD_RELATIVA','hour','xmag_disc','TEMP_norm','date_h']
M = grid[FEATURES + ['V_G1','is_real','xmag','TEMPERATURA']].dropna(subset=FEATURES + ['V_G1'])
print("filas crudas:", len(raw), "| marcas únicas:", raw.fecha.nunique(), "| rejilla 1 min:", len(M),
      "| filas reales:", int(M.is_real.sum()), "(%.0f %%)" % (100 * M.is_real.mean()))
print("rango:", M.index.min(), "->", M.index.max())
print("xmag_disc > 0: %.1f %% | TEMP_norm > 0: %.1f %% | V_G1 == 0: %.1f %%" %
      (100*(M.xmag_disc>0).mean(), 100*(M.TEMP_norm>0).mean(), 100*(M.V_G1==0).mean()))
M[FEATURES + ['V_G1']].describe().T[['mean','std','min','max']].round(3)

filas crudas: 150060 | marcas únicas: 59758 | rejilla 1 min: 90553 | filas reales: 59758 (66 %)
rango: 2022-06-17 22:37:00 -> 2022-08-19 19:49:00
xmag_disc > 0: 10.7 % | TEMP_norm > 0: 26.9 % | V_G1 == 0: 90.1 %


,mean,std,min,max
xang,115.255,101.244,0.000,359.000
xejz,-0.050,0.354,-3.190,4.030
HUMEDAD_SUELO,3514.576,946.085,67.000,4095.000
HUMEDAD_RELATIVA,46.645,6.798,30.000,68.000
hour,11.483,6.917,0.000,23.000
xmag_disc,0.437,1.286,0.000,10.105
TEMP_norm,6.921,11.467,0.000,34.700
date_h,754.108,435.676,-0.617,1508.383
V_G1,0.527,1.813,0.000,21.000


## 2. Métricas y modelos

In [3]:
def rmsle(y, p):
    return float(np.sqrt(np.mean((np.log1p(np.clip(p, 0, None)) - np.log1p(np.clip(y, 0, None))) ** 2)))

def mape_nz(y, p):
    y = np.asarray(y); p = np.asarray(p); m = y > 0
    return float(np.mean(np.abs((y[m] - p[m]) / y[m]))) if m.any() else np.nan

def metrics(y, p):
    mse = mean_squared_error(y, p)
    return dict(MAE=mean_absolute_error(y, p), MSE=mse, RMSE=math.sqrt(mse), R2=r2_score(y, p),
                RMSLE=rmsle(y, p), MAPE=mape_nz(y, p))

MODELS = {
 'Extra Trees Regressor':          lambda: ExtraTreesRegressor(n_estimators=100, n_jobs=-1, random_state=SEED),
 'Random Forest Regressor':        lambda: RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=SEED),
 'CatBoost Regressor':             lambda: CatBoostRegressor(verbose=0, random_state=SEED, thread_count=-1),
 'Extreme Gradient Boosting':      lambda: XGBRegressor(random_state=SEED, n_jobs=-1, objective='reg:squarederror'),
 'Light Gradient Boosting Machine':lambda: LGBMRegressor(random_state=SEED, verbose=-1, n_jobs=-1),
 'Gradient Boosting Regressor':    lambda: GradientBoostingRegressor(random_state=SEED),
 'K Neighbors Regressor':          lambda: KNeighborsRegressor(n_jobs=-1),
 'Decision Tree Regressor':        lambda: DecisionTreeRegressor(random_state=SEED),
 'Bayesian Ridge':                 lambda: BayesianRidge(),
 'Ridge Regression':               lambda: Ridge(random_state=SEED),
 'Linear Regression':              lambda: LinearRegression(),
 'Least Angle Regression':         lambda: Lars(),
 'Orthogonal Matching Pursuit':    lambda: OrthogonalMatchingPursuit(),
 'Huber Regressor':                lambda: HuberRegressor(),
 'Elastic Net':                    lambda: ElasticNet(random_state=SEED),
 'Lasso Regression':               lambda: Lasso(random_state=SEED),
 'AdaBoost Regressor':             lambda: AdaBoostRegressor(random_state=SEED),
 'Lasso Least Angle Regression':   lambda: LassoLars(),
 'Dummy Regressor':                lambda: DummyRegressor(),
 'Passive Aggressive Regressor':   lambda: PassiveAggressiveRegressor(random_state=SEED),
}
print(len(MODELS), "modelos")

20 modelos


## 3. Réplica: muestreo aleatorio + 10-fold (como en el paper)

90 % al azar (semilla 786) → partición interna 70/30 → 10-fold con mezcla sobre el 70 %. Debe dar un R² cercano al 0.7968 del notebook original.

In [4]:
samp = M.sample(frac=0.9, random_state=786)
tr, te = train_test_split(samp, train_size=0.7, random_state=SEED)
rows = []
for a, b in KFold(10, shuffle=True, random_state=SEED).split(tr):
    m = MODELS['Extra Trees Regressor']().fit(tr.iloc[a][FEATURES], tr.iloc[a]['V_G1'])
    rows.append(metrics(tr.iloc[b]['V_G1'], m.predict(tr.iloc[b][FEATURES])))
rep_cv = pd.DataFrame(rows)
m = MODELS['Extra Trees Regressor']().fit(tr[FEATURES], tr['V_G1'])
rep_hold = metrics(te['V_G1'], m.predict(te[FEATURES]))
print("Réplica, Extra Trees, 10-fold aleatorio (media):"); display(rep_cv.mean().round(4).to_frame('réplica'))
print("Hold-out aleatorio 30 %:"); display(pd.Series(rep_hold).round(4).to_frame('réplica'))
print("Paper (notebook original): MAE 0.1834, R2 0.7968")

Réplica, Extra Trees, 10-fold aleatorio (media):


,réplica
MAE,0.1973
MSE,0.6965
RMSE,0.8336
R2,0.7878
RMSLE,0.2584
MAPE,0.2675


Hold-out aleatorio 30 %:


,réplica
MAE,0.1830
MSE,0.6430
RMSE,0.8019
R2,0.8034
RMSLE,0.2498
MAPE,0.2428


Paper (notebook original): MAE 0.1834, R2 0.7968


## 4. Validación cronológica: 10 folds de ventana creciente

`TimeSeriesSplit(10, gap=1440)`: cada fold entrena con el pasado y prueba con el bloque siguiente, dejando **un día de separación**
para que ninguna observación vecina cruce la frontera. Se puntúa **solo sobre observaciones reales** (`is_real`), no sobre las filas generadas por `bfill`.

In [5]:
X, y, isr = M[FEATURES], M['V_G1'], M['is_real'].values
tscv = TimeSeriesSplit(n_splits=10, gap=1440)
folds = list(tscv.split(X))
for i, (a, b) in enumerate(folds[:3] + folds[-1:]):
    print("fold", i, "entrena", M.index[a[0]].date(), "->", M.index[a[-1]].date(), "| prueba", M.index[b[0]].date(), "->", M.index[b[-1]].date())

def run_cv(make, feats=FEATURES, folds=folds):
    out = []; t0 = time.time()
    for a, b in folds:
        bb = b[isr[b]]                       # solo observaciones reales
        mdl = make().fit(M.iloc[a][feats], y.iloc[a])
        out.append(metrics(y.iloc[bb], mdl.predict(M.iloc[bb][feats])))
    df_ = pd.DataFrame(out); df_.attrs['tt'] = (time.time() - t0) / len(folds)
    return df_

cv_all = {}
for name, make in MODELS.items():
    cv_all[name] = run_cv(make)
    print("%-34s R2=%.4f MAE=%.4f  (%.1f s/fold)" % (name, cv_all[name].R2.mean(), cv_all[name].MAE.mean(), cv_all[name].attrs['tt']))

fold 0 entrena 2022-06-17 -> 2022-06-22 | prueba 2022-06-23 -> 2022-06-29
fold 1 entrena 2022-06-17 -> 2022-06-28 | prueba 2022-06-29 -> 2022-07-05
fold 2 entrena 2022-06-17 -> 2022-07-04 | prueba 2022-07-05 -> 2022-07-10
fold 3 entrena 2022-06-17 -> 2022-08-13 | prueba 2022-08-14 -> 2022-08-19


Extra Trees Regressor              R2=0.3853 MAE=0.5148  (0.3 s/fold)


Random Forest Regressor            R2=0.0086 MAE=0.7943  (0.8 s/fold)


CatBoost Regressor                 R2=0.0942 MAE=0.8799  (1.9 s/fold)


Extreme Gradient Boosting          R2=-0.0624 MAE=0.8745  (0.1 s/fold)


...

K Neighbors Regressor              R2=-1.2307 MAE=0.9797  (0.0 s/fold)


Decision Tree Regressor            R2=-0.8580 MAE=0.7458  (0.1 s/fold)
Bayesian Ridge                     R2=0.3365 MAE=0.5980  (0.0 s/fold)
Ridge Regression                   R2=0.3366 MAE=0.5980  (0.0 s/fold)
Linear Regression                  R2=0.3366 MAE=0.5980  (0.0 s/fold)


Least Angle Regression             R2=0.3353 MAE=0.5868  (0.0 s/fold)
Orthogonal Matching Pursuit        R2=-0.0647 MAE=0.8460  (0.0 s/fold)


Huber Regressor                    R2=0.1312 MAE=0.3948  (0.2 s/fold)
Elastic Net                        R2=0.1453 MAE=0.7769  (0.0 s/fold)
Lasso Regression                   R2=0.0088 MAE=0.8648  (0.0 s/fold)


AdaBoost Regressor                 R2=-0.2724 MAE=1.1627  (0.7 s/fold)
Lasso Least Angle Regression       R2=0.0088 MAE=0.8648  (0.0 s/fold)
Dummy Regressor                    R2=-0.0183 MAE=0.7331  (0.0 s/fold)
Passive Aggressive Regressor       R2=-0.3010 MAE=0.9605  (0.0 s/fold)


## 5. Tabla II (cronológica): 20 modelos, media de los 10 folds

In [6]:
t2 = pd.DataFrame({n: {**d.mean().to_dict(), 'TT': d.attrs['tt']} for n, d in cv_all.items()}).T
t2 = t2.sort_values('R2', ascending=False)[['MAE','MSE','RMSE','R2','RMSLE','MAPE','TT']]
t2.round(4)

,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT
Extra Trees Regressor,0.5148,1.6516,1.2681,0.3853,0.4407,0.7098,0.2701
Linear Regression,0.5980,1.7821,1.3221,0.3366,0.4471,0.7562,0.0063
Ridge Regression,0.5980,1.7821,1.3221,0.3366,0.4471,0.7562,0.0046
Bayesian Ridge,0.5980,1.7822,1.3221,0.3365,0.4471,0.7562,0.0085
Least Angle Regression,0.5868,1.7859,1.3234,0.3353,0.4440,0.7685,0.0045
Light Gradient Boosting Machine,0.6547,2.0477,1.3818,0.2349,0.4903,0.7565,0.2044
Elastic Net,0.7769,2.2776,1.4960,0.1453,0.5205,0.7449,0.0056
Huber Regressor,0.3948,2.3009,1.5031,0.1312,0.4397,0.9321,0.1582
Gradient Boosting Regressor,0.7480,2.3686,1.4398,0.1121,0.5123,0.7550,2.8184
CatBoost Regressor,0.8799,2.4261,1.5295,0.0942,0.6012,0.7660,1.9246


In [7]:
def f4(x): return ("-" if x < 0 else "") + "%.4f" % abs(x)
lines = []
for i, (n, r) in enumerate(t2.iterrows()):
    lines.append("        %s & %s \\\\" % (n, " & ".join(f4(r[c]) for c in ['MAE','MSE','RMSE','R2','RMSLE','MAPE','TT'])))
    if i == 0: lines.append("        \\hline")
tabla2 = "\n".join(lines)
(OUT / "tabla_II_cronologica.tex").write_text(tabla2)
t2.round(4).to_csv(OUT / "tabla_II_cronologica.csv")
print(tabla2)

        Extra Trees Regressor & 0.5148 & 1.6516 & 1.2681 & 0.3853 & 0.4407 & 0.7098 & 0.2701 \\
        \hline
        Linear Regression & 0.5980 & 1.7821 & 1.3221 & 0.3366 & 0.4471 & 0.7562 & 0.0063 \\
        Ridge Regression & 0.5980 & 1.7821 & 1.3221 & 0.3366 & 0.4471 & 0.7562 & 0.0046 \\
        Bayesian Ridge & 0.5980 & 1.7822 & 1.3221 & 0.3365 & 0.4471 & 0.7562 & 0.0085 \\
        Least Angle Regression & 0.5868 & 1.7859 & 1.3234 & 0.3353 & 0.4440 & 0.7685 & 0.0045 \\
        Light Gradient Boosting Machine & 0.6547 & 2.0477 & 1.3818 & 0.2349 & 0.4903 & 0.7565 & 0.2044 \\
        Elastic Net & 0.7769 & 2.2776 & 1.4960 & 0.1453 & 0.5205 & 0.7449 & 0.0056 \\
        Huber Regressor & 0.3948 & 2.3009 & 1.5031 & 0.1312 & 0.4397 & 0.9321 & 0.1582 \\
        Gradient Boosting Regressor & 0.7480 & 2.3686 & 1.4398 & 0.1121 & 0.5123 & 0.7550 & 2.8184 \\
        CatBoost Regressor & 0.8799 & 2.4261 & 1.5295 & 0.0942 & 0.6012 & 0.7660 & 1.9246 \\
        Lasso Least Angle Regression & 0.86

## 6. Tabla V (cronológica): Extra Trees fold a fold

In [8]:
et = cv_all['Extra Trees Regressor'][['MAE','MSE','RMSE','R2','RMSLE','MAPE']]
rows = ["        %d & %s \\\\" % (i, " & ".join(f4(v) for v in r)) for i, r in et.iterrows()]
mean, std = et.mean(), et.std()
rows += ["        \\hline",
         "        \\textbf{Mean} & " + " & ".join("\\textbf{%s}" % f4(v) for v in mean) + " \\\\",
         "        \\textbf{Std} & " + " & ".join(f4(v) for v in std) + " \\\\"]
tabla5 = "\n".join(rows)
(OUT / "tabla_V_cronologica.tex").write_text(tabla5)
et.round(4).to_csv(OUT / "tabla_V_cronologica.csv")
print(tabla5)

        0 & 0.3268 & 1.0340 & 1.0168 & 0.4541 & 0.3471 & 0.6568 \\
        1 & 0.5481 & 1.6564 & 1.2870 & 0.4629 & 0.4653 & 0.6744 \\
        2 & 0.6283 & 1.7756 & 1.3325 & 0.1776 & 0.5215 & 0.6750 \\
        3 & 0.5559 & 1.8723 & 1.3683 & 0.2611 & 0.4619 & 0.7608 \\
        4 & 0.8321 & 1.9204 & 1.3858 & 0.1107 & 0.6091 & 0.6826 \\
        5 & 0.2566 & 0.7756 & 0.8807 & 0.5730 & 0.2846 & 0.7177 \\
        6 & 0.3321 & 1.2611 & 1.1230 & 0.3825 & 0.3537 & 0.7991 \\
        7 & 0.4437 & 1.4391 & 1.1996 & 0.4749 & 0.4237 & 0.7419 \\
        8 & 0.6018 & 2.6700 & 1.6340 & 0.5292 & 0.4383 & 0.6399 \\
        9 & 0.6227 & 2.1117 & 1.4532 & 0.4268 & 0.5017 & 0.7502 \\
        \hline
        \textbf{Mean} & \textbf{0.5148} & \textbf{1.6516} & \textbf{1.2681} & \textbf{0.3853} & \textbf{0.4407} & \textbf{0.7098} \\
        \textbf{Std} & 0.1751 & 0.5509 & 0.2200 & 0.1529 & 0.0947 & 0.0519 \\


## 7. ¿Qué aporta cada variable? Ablaciones con el mismo esquema cronológico

`date_h` (fecha truncada a la hora) funciona como un reloj: con validación aleatoria permite "memorizar" el tiempo; en una prueba futura queda fuera del rango visto.

In [9]:
abl = {
  'Las 8 variables del paper': FEATURES,
  'Sin date_h': [f for f in FEATURES if f != 'date_h'],
  'Sin HUMEDAD_SUELO': [f for f in FEATURES if f != 'HUMEDAD_SUELO'],
  'Sin date_h ni HUMEDAD_SUELO': [f for f in FEATURES if f not in ('date_h','HUMEDAD_SUELO')],
}
res = {k: run_cv(MODELS['Extra Trees Regressor'], feats=v).mean() for k, v in abl.items()}
# viento y temperatura continuos en lugar de discretizados
M2 = M.copy(); M2['xmag_c'] = M2['xmag']; M2['temp_c'] = M2['TEMPERATURA']
feats_c = [f for f in FEATURES if f not in ('date_h','xmag_disc','TEMP_norm')] + ['xmag_c','temp_c']
def run_cv_M2(make, feats):
    out = []
    for a, b in folds:
        bb = b[isr[b]]; mdl = make().fit(M2.iloc[a][feats], y.iloc[a]); out.append(metrics(y.iloc[bb], mdl.predict(M2.iloc[bb][feats])))
    return pd.DataFrame(out)
res['Continuas (viento, temp.), sin date_h'] = run_cv_M2(MODELS['Extra Trees Regressor'], feats_c).mean()
pd.DataFrame(res).T[['MAE','RMSE','R2','RMSLE']].round(4)

,MAE,RMSE,R2,RMSLE
Las 8 variables del paper,0.5148,1.2681,0.3853,0.4407
Sin date_h,0.3933,1.2115,0.4475,0.3761
Sin HUMEDAD_SUELO,0.4995,1.2756,0.3760,0.4383
Sin date_h ni HUMEDAD_SUELO,0.3913,1.2106,0.4459,0.3807
"Continuas (viento, temp.), sin date_h",0.3909,1.1865,0.4682,0.3643


## 8. Comparación final: aleatorio frente a cronológico (Extra Trees)

In [10]:
cmp_ = pd.DataFrame({
  'Réplica: 10-fold aleatorio (paper)': rep_cv.mean(),
  'Réplica: hold-out aleatorio 30 %': pd.Series(rep_hold),
  'Cronológico: 10 folds (obs. reales)': cv_all['Extra Trees Regressor'].mean(),
}).T[['MAE','MSE','RMSE','R2','RMSLE','MAPE']]
display(cmp_.round(4))
cmp_.round(4).to_csv(OUT / "comparacion_aleatorio_vs_cronologico.csv")
(OUT / "resumen_experimento.json").write_text(json.dumps({
    "filas_crudas": int(len(raw)), "marcas_unicas": int(raw.fecha.nunique()), "filas_rejilla_1min": int(len(M)),
    "filas_reales": int(M.is_real.sum()), "rango": [str(M.index.min()), str(M.index.max())],
    "pct_xmag_disc_positivo": round(float(100*(M.xmag_disc>0).mean()), 2),
    "pct_temp_norm_positivo": round(float(100*(M.TEMP_norm>0).mean()), 2),
    "esquema": "TimeSeriesSplit(10, gap=1440), puntuado solo en observaciones reales",
    "versiones": {"python": platform.python_version(), "sklearn": sklearn.__version__, "pandas": pd.__version__}}, indent=2))

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Réplica: 10-fold aleatorio (paper),0.1973,0.6965,0.8336,0.7878,0.2584,0.2675
Réplica: hold-out aleatorio 30 %,0.1830,0.6430,0.8019,0.8034,0.2498,0.2428
Cronológico: 10 folds (obs. reales),0.5148,1.6516,1.2681,0.3853,0.4407,0.7098


429